# `@wasm` — every use case, run (v0.2.5 M1.5 full-features notebook)

One `@wasm` decorator on ordinary Python; this notebook **runs every row** of the use-case tables (server, client, fallback) and the two honest "where it loses" cases. Every number below is **computed by `features_lib.py` from the inputs set in this notebook** — nothing is typed in — and the last cell proves it by changing an input and watching the table change.

Setup (from the repository root):

```bash
pip install -e .[test]                      # pythscribe + wasmtime + gradio + playwright + numpy ...
python -m playwright install chromium       # for the in-tab (isomorphic) cell only
python -m pythscribe.build examples/wasm-use-cases/kernels.py
jupyter nbconvert --to notebook --execute examples/wasm-use-cases/full_features.ipynb
```

**Positioning, stated up front:** on the server, `@wasm` competes with NumPy, Numba and Cython, not with nothing. Its wins are the cases those *cannot* do — non-vectorisable loops with Python semantics, a capability sandbox with fuel metering, GIL-free fan-out without pickling, bit-for-bit determinism across machines and across browser/server, one artifact — **not raw speed**. The speedup below is against *interpreted CPython* and is reported that way; the vectorised-NumPy control shows NumPy winning, as it should. (Numba's `@njit` covers part of the first row; it rejects the sandbox, the browser, and Python semantics such as big ints and exceptions.)

In [1]:
import os, sys, json, time
from pathlib import Path
from IPython.display import Markdown, display
HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import features_lib as F

FAST = os.environ.get("NB_FAST", "0") == "1"      # the test runner shrinks the sizes; the numbers are still measured
K = F.load_kernels(HERE / "kernels.py")
modes = {name: (F.binding(getattr(K, name)).mode, F.binding(getattr(K, name)).mode_reason)
         for name in ("edit_distance", "dtw_distance", "viterbi", "mask_digit_runs", "spin", "sum_squares", "is_vowel", "count_vowels")}
for name, (mode, why) in modes.items():
    print(f"{name:16s} mode={mode:8s} {why}")
assert all(m == "server" for m, _ in modes.values()), "build the artifacts first: python -m pythscribe.build examples/wasm-use-cases/kernels.py (and pip install wasmtime)"
records = {}

edit_distance    mode=server   auto: artifact + wasmtime + FFI grammar
dtw_distance     mode=server   auto: artifact + wasmtime + FFI grammar
viterbi          mode=server   auto: artifact + wasmtime + FFI grammar
mask_digit_runs  mode=server   auto: artifact + wasmtime + FFI grammar
spin             mode=server   auto: artifact + wasmtime + FFI grammar
sum_squares      mode=server   auto: artifact + wasmtime + FFI grammar
is_vowel         mode=server   auto: artifact + wasmtime + FFI grammar
count_vowels     mode=server   auto: artifact + wasmtime + FFI grammar


## Server 1 — non-vectorisable loops: edit distance (DP) vs interpreted CPython

`edit_distance` is a rolling-row Levenshtein DP: data-dependent branches, no vectorisation. Both paths run the *same function* on the same inputs and are checked against an independent full-matrix reference.

In [2]:
records["speedup"] = F.measure_speedup(K, n=200 if FAST else 800, seed=0, repeats=3)
r = records["speedup"]
print(f"n={r['n']}x{r['n']}: WASM in-process {r['wasm_s']*1e3:.1f} ms vs CPython {r['cpython_s']*1e3:.0f} ms -> {r['ratio']:.1f}x   (all three agree: {r['all_equal']})")
print("baseline:", r["baseline"]); print("host imports of the .wasm:", r["host_imports"] or "none (pure WASM)")

n=800x800: WASM in-process 12.0 ms vs CPython 164 ms -> 13.7x   (all three agree: True)
baseline: interpreted CPython (the same function's Python body) -- NOT NumPy/Numba
host imports of the .wasm: none (pure WASM)


## Server 2 — sandboxed (LLM-generated) code: fuel + no ambient I/O, with the escape attempts

A representative generated snippet is compiled by `pyths` and run under `Sandbox(fuel=…)`. Then the **paired negative controls** — the ones that make the claim mean something:

* an **unbounded loop** must hit the fuel trap (RED half: the same loop, unmetered, runs until an external epoch interrupt kills it — so it really is unbounded);
* a module that **imports WASI file I/O** must be refused at link time (RED half: the same module, under a plain wasmtime linker *with* WASI, writes a file — so it really is an I/O attempt);
* a snippet calling `open()` never becomes a `.wasm` artifact.

**This cell fails if any escape succeeds** (`assert rec["contained"]`).

In [3]:
records["sandbox"] = F.measure_sandbox(K, fuel=20_000_000, workdir=HERE / "_sandbox_work")
sb = records["sandbox"]
print("snippet:", sb["snippet"])
print("fuel trap:", {k: v for k, v in sb["fuel_trap"].items() if k != "error"})
print("   error:", sb["fuel_trap"]["error"])
print("I/O import:", sb["io_attempt"]["refused"], "-", sb["io_attempt"]["error"][:110], "...")
print("   RED half (caller-built WASI linker):", sb["io_attempt"]["red_control"])
print("open() in a kernel:", sb["open_in_kernel"])
assert sb["contained"], "AN ESCAPE SUCCEEDED -- the sandbox claim is false"
print("\ncontained =", sb["contained"], "(every positive AND every RED half held)")

snippet: {'value': 342583, 'cpython_value': 342583, 'equal': True, 'fuel_used': 2250075, 'host_imports': [], 'wasm_bytes': 766}
fuel trap: {'trapped': True, 'elapsed_s': 0.0019428000086918473, 'finite_call_fuel_used': 9, 'red_control': {'ran_for_s': 0.30058229994028807, 'outcome': 'interrupted by epoch after 0.30s (INTERRUPT)', 'still_running_at_kill': True}}
   error: spin: fuel budget of 20000000 exhausted (execution bounded by the sandbox)
I/O import: True - io_attempt: import `wasi_snapshot_preview1.path_open` refused -- the sandbox grants no capability beyond the p ...
   RED half (caller-built WASI linker): {'wrote_file_under_wasi_linker': True, 'content': 'pwned\n'}
open() in a kernel: {'refused_at_build': True, 'error': 'pyths compile failed for `leak`:', 'artifact_exists': False}

contained = True (every positive AND every RED half held)


## Server 3 — GIL-free fan-out

Each thread gets its own wasmtime instance and the ctypes call releases the GIL. The **control** is the same threads running the CPython body: the GIL serialises them.

In [4]:
records["fanout"] = F.measure_fanout(K, threads=4, n=800 if FAST else 1200, seed=0, reps=2)
fo = records["fanout"]
print(f"{fo['threads']} threads on {fo['cpu_count']} CPUs: WASM {fo['wasm_single_s']*1e3:.0f} ms single -> {fo['wasm_parallel_s']*1e3:.0f} ms parallel = {fo['wasm_scaling']:.2f}x scaling")
print(f"CPython control:               {fo['cpython_single_s']*1e3:.0f} ms single -> {fo['cpython_parallel_s']*1e3:.0f} ms parallel = {fo['cpython_scaling']:.2f}x scaling (the GIL)")

4 threads on 16 CPUs: WASM 52 ms single -> 64 ms parallel = 3.26x scaling
CPython control:               802 ms single -> 3135 ms parallel = 1.02x scaling (the GIL)


## Server 4 — bit-for-bit determinism, across engines

Same `.wasm`, same bytes every run (`sha256` of the Viterbi score + path), equal to the CPython body, and equal to the **browser's own JS shim under Node/V8** — three engines, one digest. This holds *by construction* for kernels with no host imports (pure i64/f64 WASM); a kernel that calls `math.sin` & co. inherits the host libm, and the runtime says so (`host_imports`).

In [5]:
records["determinism"] = F.measure_determinism(K, runs=5, n_states=6, n_steps=100 if FAST else 400, seed=0)
de = records["determinism"]
print("digests over", de["runs"], "runs:", sorted(set(de["digests"])))
print("== CPython:", de["equal_cpython"], "  == Node/V8 shim:", de["equal_node_v8"], "  host imports:", de["host_imports"] or "none")

digests over 5 runs: ['ffe2af0218263324216b0c3e250383a92a0c65155168dc9d7ae854d08636dd36']
== CPython: True   == Node/V8 shim: True   host imports: none


## Server 5 — single artifact, and the fallback

Only the `.wasm` is needed at run time: copy that one file elsewhere, make the compiler unreachable, run. And the same module with **no** artifact at all runs as plain Python — same answer, `mode='fallback'` (safe to try).

In [6]:
records["single_artifact"] = F.measure_single_artifact(K, HERE / "_single_work")
records["fallback"] = F.measure_fallback(HERE / "_fallback_work")
print("single artifact:", records["single_artifact"])
print("fallback:", {k: v for k, v in records["fallback"].items() if k != "mode_reason"})

single artifact: {'kind': 'single_artifact', 'wasm_bytes': 1861, 'files_needed': 1, 'compiler_reachable': False, 'value': 3, 'expected': 3, 'ok': True}
fallback: {'kind': 'fallback', 'mode': 'fallback', 'artifact_status': 'absent', 'value': 3, 'expected': 3, 'python_calls': 1, 'server_calls': 0, 'ok': True}


## Client — isomorphic (a real tab), zero round-trip, on-device redaction, preprocessing

* **Isomorphic / zero-round-trip:** `app.py` is launched and driven in a headless Chromium (Playwright). The tab runs `dtw_distance` through the M1 Gradio component; the server ran the *same function* in-process (wasmtime) and as CPython *before* dispatch. The three IEEE-754 bit patterns must be identical, and the server-side counters since dispatch must both be **0** (the tab did the work). Needs `gradio` + `playwright`; recorded as NOT RUN otherwise.
* **On-device redaction:** `mask_digit_runs` through the browser's own shim under Node (V8), the server path, and CPython — masked buffers byte-identical, and equal to an independent regex reference. (Stated plainly: that is the browser *channel*, not a tab; the tab run is the isomorphic cell.)
* **Preprocessing at the edge:** M1's committed evidence, re-read, not restated.

In [7]:
try:
    import iso_drive as drive
    records["isomorphic"] = drive.measure_isomorphic([0.0, 1.5, 2.25, 3.0, 2.5, 1.0, -0.5, 0.1], [0.0, 0.2, 1.0, 2.0, 3.1, 3.0, 2.0, 1.0, 0.0, -0.1], app_dir=HERE)
    iso = records["isomorphic"]
    print("isomorphic:", {k: iso[k] for k in ("path", "browser_bits", "server_bits", "cpython_bits", "identical", "python_calls", "server_calls", "wasm_fetched", "mode")})
    assert iso["console_errors"] == [], iso["console_errors"]
except ImportError as e:
    records["isomorphic"] = None
    print("isomorphic cell NOT RUN:", e)
records["redaction"] = F.measure_redaction_via_browser_shim(K, seed=0, n=4000, min_run=4)
rd = records["redaction"]
print("redaction:", {k: rd[k] for k in ("masked_server", "masked_cpython", "masked_browser_shim", "all_identical", "matches_regex_reference")})
print("   before:", rd["sample_before"]); print("   after: ", rd["sample_after"])
records["m1_preprocessing"] = F.m1_preprocessing_evidence()
print("M1 preprocessing evidence:", records["m1_preprocessing"])

isomorphic: {'path': 'browser-wasm', 'browser_bits': '0000000000000240', 'server_bits': '0000000000000240', 'cpython_bits': '0000000000000240', 'identical': True, 'python_calls': 0, 'server_calls': 0, 'wasm_fetched': True, 'mode': 'server'}
redaction: {'masked_server': 1122, 'masked_cpython': 1122, 'masked_browser_shim': 1122, 'all_identical': True, 'matches_regex_reference': True}
   before: tel 679215ref 521760889tel 62468ref 9528138969id 18316call 7
   after:  tel ******ref *********tel *****ref **********id *****call *
M1 preprocessing evidence: {'kind': 'm1_preprocessing', 'aggregate_reduction_x': 88.85357920806851, 'images': ['photo_640x480.jpg', 'photo_1600x1200.jpg', 'photo_4000x3000.jpg'], 'client_paths': 'browser-wasm', 'source': 'examples/gradio-image-preprocess/metrics_summary.json'}


## Where it loses — say so on the page

1. **Already-vectorised code.** A sum of squares over a million floats: NumPy is C already, and `@wasm` also pays to marshal the list across the boundary. NumPy wins by hundreds of times -- and the marshaling can make `@wasm` slower than the interpreter itself here (the table says which).
2. **Per-element boundary crossing.** Calling a scalar kernel once per character is marshaling-dominated; one batched call over the same data is orders of magnitude cheaper. Batch the call.

In [8]:
records["where_it_loses"] = F.measure_where_it_loses(K, n=100_000 if FAST else 1_000_000, per_element_n=5_000 if FAST else 20_000, seed=0)
wl = records["where_it_loses"]
v, pe = wl["vectorised"], wl["per_element"]
print(f"sum of squares n={wl['n']}: NumPy {v['numpy_s']*1e3:.2f} ms | @wasm {v['wasm_s']*1e3:.1f} ms | CPython {v['cpython_s']*1e3:.0f} ms  -> NumPy wins by {v['numpy_over_wasm_x']:.0f}x ({v['numpy_wins']})")
print(f"per-element: {wl['per_element_n']} scalar calls {pe['per_call_s']*1e3:.0f} ms ({pe['per_call_us']:.1f} us/call) vs one batched call {pe['batched_s']*1e3:.2f} ms -> {pe['per_element_over_batched_x']:.0f}x  (batching wins: {pe['batching_wins']})")

sum of squares n=1000000: NumPy 0.60 ms | @wasm 234.9 ms | CPython 44 ms  -> NumPy wins by 394x (True)
per-element: 20000 scalar calls 2261 ms (113.1 us/call) vs one batched call 3.96 ms -> 572x  (batching wins: True)


## The table — every cell derived from the records above

In [9]:
summary = F.summarize(records)
display(Markdown(F.format_table(summary)))
F.save_evidence(records, summary, HERE / "full_features_records.json", HERE / "full_features_summary.json")
print("saved full_features_records.json + full_features_summary.json")

| Use case | Claim | Measured | Pass |
|---|---|---|---|
| Non-vectorizable loops (server) | same answer on every path; speedup vs interpreted CPython REPORTED | 13.7x vs CPython on 800x800 edit distance (12.0 ms vs 164 ms); all paths == reference True | YES |
| Sandboxed (LLM-generated) code (server) | fuel-bounded, no ambient I/O; escapes are RED | snippet ok (fuel used 2250075); loop trapped in 2 ms; WASI import refused; open() refused at build; RED halves: unmetered loop ran until killed=True, WASI linker wrote file=True | YES |
| GIL-free parallelism (server) | N threads scale; the CPython control does not | 4 threads on 16 CPUs: WASM scaling 3.26x, CPython control 1.02x | YES |
| Bit-for-bit determinism (server) | same bytes every run, == CPython, == V8 shim | 5 runs, 1 digest(s); == CPython True; == Node/V8 True; host imports none | YES |
| Single-artifact deployment (server) | one .wasm, no toolchain at run time | 1861 B .wasm ran with the compiler unreachable (3 == 3) | YES |
| Isomorphic same-fn (browser + server) | one @wasm fn, tab and in-process, identical bits | browser bits 0000000000000240 == server bits 0000000000000240 == CPython 0000000000000240: True (mode=server; browser python_calls=0, server_calls=0) | YES |
| Zero-round-trip interactivity (browser) | the tab computes; the server does nothing | path=browser-wasm, server python_calls=0, server_calls=0, .wasm fetched by the tab=True | YES |
| Preprocessing at the edge (browser) | resize in the tab before upload | 89x fewer bytes uploaded across 3 images (M1 evidence, examples/gradio-image-preprocess/metrics_summary.json) | YES |
| On-device redaction (browser channel) | PII masked before upload; identical everywhere; == regex spec | 1122 of 1960 chars masked; V8 shim / server / CPython digests identical=True; == regex reference True | YES |
| Fallback (no artifact) | plain Python, same answer | mode=fallback (absent); value 3 == 3; python_calls=1 | YES |
| WHERE IT LOSES: already-vectorised code | NumPy (C) is not beaten -- say so (and the answers agree) | sum of squares, n=1000000: NumPy 0.60 ms vs @wasm 234.9 ms vs CPython 44 ms (NumPy 394x faster; @wasm even 5x SLOWER than CPython here -- marshaling 1000000 floats across the boundary dominates); values agree True | YES |
| WHERE IT LOSES: per-element boundary crossing | marshaling dominates -- batch the call | 20000 per-element calls 2261 ms (113.1 us/call) vs one batched call 3.96 ms (572x) | YES |

12/12 rows measured; all measured rows pass: **True** (numbers computed, not constants)

saved full_features_records.json + full_features_summary.json


### Control: the numbers are computed, not constants

Halve the measured WASM time in the speedup record and the table's first row changes; drop a record and `summarize` refuses. If the table were typed in, neither would happen.

In [10]:
import copy
mut = copy.deepcopy(records); mut["speedup"]["wasm_s"] *= 0.5
assert F.format_table(F.summarize(mut)) != F.format_table(summary)
assert F.summarize(mut)["rows"][0]["measured"] != summary["rows"][0]["measured"]
missing = copy.deepcopy(records); del missing["fanout"]
try:
    F.summarize(missing); raise SystemExit("summarize accepted a missing record")
except ValueError as e:
    print("missing record refused:", e)
print("computed, not constants: a changed input changes the table")
print("all measured rows pass:", summary["all_pass"], f"({summary['n_measured']}/{summary['n_rows']} measured)")
assert summary["all_pass"]
assert summary["n_measured"] == summary["n_rows"], "this is the EVIDENCE run: every row must be measured, not NOT RUN (opus m1.5 r2/NEW-6)"

missing record refused: record 'fanout' missing: the notebook must measure it
computed, not constants: a changed input changes the table
all measured rows pass: True (12/12 measured)


## Caveats, honestly

* The speedup is **vs interpreted CPython**. Numba's `@njit` would also compile the DP row (and is mature); it does not give you the sandbox, the browser, or Python's semantics for big ints, exceptions and classes. We do not headline a single ratio.
* Bit-for-bit identity across engines is by construction for **pure** kernels (no host imports). With `math.*` calls, the browser's V8 libm and CPython's platform libm may differ in the last ulp for transcendental functions; `binding.server.host_imports` tells you which kernels depend on the host.
* The redaction row ran the browser's *shim* under Node; only the isomorphic row ran a real tab.
* pyths 0.2.4 evaluates both operands of `and`/`or` in the WASM backend (no short-circuit) — an upstream bug found while writing `mask_digit_runs`; the kernel avoids the idiom (`pythscribe/review/2026-09-03-upstream-bug-wasm-short-circuit-and.md`).